In [1]:
import sys
import os
from torch.utils.data import DataLoader

# Add the parent directory to the system path so Python can locate the 'src' folder
sys.path.append(os.path.abspath(".."))

# Import the dataset module securely from your local repository
from src.dataset import EnzymeSubstrateDataset, collate_fn

# Check environment routing to ensure smooth cross-platform path mapping
if os.path.exists("../datasets"):
    base_path = ".."
elif os.path.exists("datasets"):
    base_path = "."
else:
    raise FileNotFoundError("Could not locate 'datasets' folder in this workspace.")

# 1. Setup absolute correct file paths based directly on verified file logs
data_file = f"{base_path}/datasets/processed/kcat_max_wt_singleSeqs_wpdbs.csv"
train_split = f"{base_path}/datasets/splits/kcat-random_train.csv" # Fixed file syntax layout

# 2. Test initialization and filter verification
print("--- Testing Dataset Split Loading ---")
try:
    dataset = EnzymeSubstrateDataset(data_path=data_file, split_path=train_split)
    print(f"Success! Total in-distribution training samples loaded: {len(dataset)}")
except Exception as e:
    print(f"Error during dataset initialization: {e}")

# 3. Test fetching a single data point
print("\n--- Testing Single Item Retrieval ---")
try:
    sequence, smiles, target = dataset[0]
    print(f"Sequence (first 30 residues): {sequence[:30]}...")
    print(f"SMILES Structure String     : {smiles}")
    print(f"Target Label (log10 value)  : {target.item():.4f}")
except Exception as e:
    print(f"Error during fetching single item: {e}")

# 4. Test DataLoader batch aggregation
print("\n--- Testing DataLoader Batching ---")
try:
    loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
    batch_seqs, batch_smiles, batch_targets = next(iter(loader))
    
    print(f"Batched sequences count: {len(batch_seqs)} (Expected: 4)")
    print(f"Batched smiles count   : {len(batch_smiles)} (Expected: 4)")
    print(f"Batched targets shape  : {batch_targets.shape} (Expected: torch.Size([4]))")
    print("\nData pipeline check passed! Dataset module is working flawlessly.")
except Exception as e:
    print(f"Error during DataLoader batch aggregation: {e}")

--- Testing Dataset Split Loading ---
Success! Total in-distribution training samples loaded: 18509

--- Testing Single Item Retrieval ---
Sequence (first 30 residues): MAAFSLSAKQILSPSTHRPSLSKTTTADSS...
SMILES Structure String     : >>O=P(O)(O)O
Target Label (log10 value)  : -0.0458

--- Testing DataLoader Batching ---
Batched sequences count: 4 (Expected: 4)
Batched smiles count   : 4 (Expected: 4)
Batched targets shape  : torch.Size([4]) (Expected: torch.Size([4]))

Data pipeline check passed! Dataset module is working flawlessly.


In [2]:
import sys
import os
import torch
from torch.utils.data import DataLoader

# 1. Environment configuration: Add parent directory to system path
sys.path.append(os.path.abspath(".."))
from src.dataset import EnzymeSubstrateDataset, collate_fn
from src.encoders import SequenceEncoder, SubstrateEncoder

# Dynamic path resolution to handle varying Jupyter working directories
if os.path.exists("../datasets"):
    base_path = ".."
else:
    base_path = "."

data_file = f"{base_path}/datasets/processed/kcat_max_wt_singleSeqs_wpdbs.csv"
train_split = f"{base_path}/datasets/splits/kcat-random_train.csv"

# 2. Data fetching
print("[Check Point 1] Preparing to load data batch...")
dataset = EnzymeSubstrateDataset(data_path=data_file, split_path=train_split)
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
batch_seqs, batch_smiles, batch_targets = next(iter(loader))
print(f"[Success] Data batch loaded! Seq length: {len(batch_seqs[0])}, SMILES: {batch_smiles[0]}")

# 3. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[Check Point 2] Compute device confirmed: {device}")

# 4. Load Models (Tracking instantiation progress)
print("\n[Check Point 3] Loading ESM-2 protein model...")
seq_encoder = SequenceEncoder().to(device)
print("[Success] ESM-2 loaded successfully!")

print("\n[Check Point 4] Loading ChemBERTa substrate model...")
print("-> Note: This may take a moment to download weights if not cached.")
sub_encoder = SubstrateEncoder().to(device)
print("[Success] ChemBERTa loaded successfully!")

# 5. Execute Feature Extraction (Forward Pass)
print("\n[Check Point 5] Extracting protein features via ESM-2...")
with torch.no_grad():
    protein_vectors = seq_encoder(batch_seqs, device)
print(f"[Success] Protein features extracted! Tensor shape: {list(protein_vectors.shape)}")

print("\n[Check Point 6] Extracting substrate features via ChemBERTa...")
with torch.no_grad():
    substrate_vectors = sub_encoder(batch_smiles, device)
print(f"[Success] Substrate features extracted! Tensor shape: {list(substrate_vectors.shape)}")

print("\n🎉 Jupyter test passed completely! Feature engine is fully operational.")

c:\Users\23604\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Check Point 1] Preparing to load data batch...
[Success] Data batch loaded! Seq length: 697, SMILES: O.O=[N+]([O-])c1ccc(O[C@@H]2OC[C@@H](O)[C@H](O)[C@H]2O)cc1>>O=[N+]([O-])c1ccc(O)cc1.OC1COC(O)C(O)C1O

[Check Point 2] Compute device confirmed: cpu

[Check Point 3] Loading ESM-2 protein model...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 12060.07it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[Success] ESM-2 loaded successfully!

[Check Point 4] Loading ChemBERTa substrate model...
-> Note: This may take a moment to download weights if not cached.


Loading weights: 100%|██████████| 53/53 [00:00<00:00, 25987.62it/s]
[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
regression.out_proj.bias   | UNEXPECTED | 
norm_std                   | UNEXPECTED | 
norm_mean                  | UNEXPECTED | 
regression.out_proj.weight | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[Success] ChemBERTa loaded successfully!

[Check Point 5] Extracting protein features via ESM-2...
[Success] Protein features extracted! Tensor shape: [2, 320]

[Check Point 6] Extracting substrate features via ChemBERTa...
[Success] Substrate features extracted! Tensor shape: [2, 384]

🎉 Jupyter test passed completely! Feature engine is fully operational.


In [4]:
import sys
import os
import torch
from torch.utils.data import DataLoader

# Add parent directory to path
sys.path.append(os.path.abspath(".."))

from src.dataset import EnzymeSubstrateDataset, collate_fn
from src.models import HeteroscedasticEnzymeModel
from src.losses import HeteroscedasticGaussianNLLLoss

# Dynamic path resolution
if os.path.exists("../datasets"):
    base_path = ".."
else:
    base_path = "."

data_file = f"{base_path}/datasets/processed/kcat_max_wt_singleSeqs_wpdbs.csv"
train_split = f"{base_path}/datasets/splits/kcat-random_train.csv"

# [Check Point 1] Load Data
print("[Check Point 1] Loading data batch...")
dataset = EnzymeSubstrateDataset(data_path=data_file, split_path=train_split)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
batch_seqs, batch_smiles, batch_targets = next(iter(loader))
print(f"[Success] Batch size: {len(batch_seqs)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_targets = batch_targets.to(device)

# [Check Point 2] Initialize Full Model & Loss
print("\n[Check Point 2] Initializing the full Heteroscedastic Model...")
model = HeteroscedasticEnzymeModel().to(device)
criterion = HeteroscedasticGaussianNLLLoss()
print("[Success] Model and Loss function instantiated!")

# [Check Point 3] Forward Pass
print("\n[Check Point 3] Executing Multi-Modal Forward Pass...")
with torch.no_grad():
    predicted_means, predicted_logvars = model(batch_seqs, batch_smiles, device)
    
    # Calculate initial loss
    loss = criterion(predicted_means, predicted_logvars, batch_targets)

print(f"[Success] Forward pass complete!")
print(f"-> Predicted Means Shape    : {list(predicted_means.shape)}")
print(f"-> Predicted Log-Vars Shape : {list(predicted_logvars.shape)}")

# [Check Point 4] Loss Verification
print("\n[Check Point 4] Calculating Initial Loss...")
print(f"-> Ground Truth Targets : {batch_targets.cpu().numpy().round(4)}")
print(f"-> Initial Model Means  : {predicted_means.cpu().numpy().round(4)}")
print(f"-> Initial Model Log-Vars: {predicted_logvars.cpu().numpy().round(4)}")
print(f"-> Computed NLL Loss    : {loss.item():.4f}")

print("\n🚀 Step 3 Check Passed! The Multimodal Brain is fully operational.")

[Check Point 1] Loading data batch...
[Success] Batch size: 4

[Check Point 2] Initializing the full Heteroscedastic Model...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 8561.00it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 26404.34it/s]
[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
regression

[Success] Model and Loss function instantiated!

[Check Point 3] Executing Multi-Modal Forward Pass...
[Success] Forward pass complete!
-> Predicted Means Shape    : [4]
-> Predicted Log-Vars Shape : [4]

[Check Point 4] Calculating Initial Loss...
-> Ground Truth Targets : [-0.284   2.0678  1.5726  3.918 ]
-> Initial Model Means  : [-0.0587  0.031  -0.0497  0.0764]
-> Initial Model Log-Vars: [ 0.0227 -0.0528  0.0424 -0.0088]
-> Computed NLL Loss    : 2.7297

🚀 Step 3 Check Passed! The Multimodal Brain is fully operational.
